# Imports and project loading

In [2]:
from pyiron_atomistics import Project
import pandas as pd
import numpy as np
from pathlib import Path

# Your pyiron project (adjust if needed)
pr = Project("../cu_zr_mlip_project")

print("Project path:", pr.path)

Project path: /Users/ksenis/pyiron/projects/CuZr/cu_zr_mlip_project/


# Global setting

In [3]:
# MACE model file is stored in pyiron resources: .../lammps/potentials/
MACE_MODEL_FILE = "CuZr_mace_debug.model-lammps.pt"

# Default compute settings (good for laptop + avoids oversubscription)
DEFAULT_CORES = 1

# MD defaults
DEFAULT_TIMESTEP_FS = 1.0
DEFAULT_THERMO_STEPS = 50

# Build the MACE “custom potential” DataFrame

In [4]:
def make_mace_potential_df(model_file: str, elements=("Cu", "Zr")) -> pd.DataFrame:
    el = list(elements)
    return pd.DataFrame({
        "Name": ["MACE_CuZr"],
        "Filename": [[model_file]],     # resolved via pyiron resources
        "Model": ["Custom"],
        "Species": [el],
        "Config": [[
            "pair_style mace\n",
            f"pair_coeff * * {model_file} " + " ".join(el) + "\n",
            # Optional sanity print in log.lammps:
            # "info pair\n",
        ]],
    })

MACE_POTENTIAL_DF = make_mace_potential_df(MACE_MODEL_FILE, elements=("Cu", "Zr"))
MACE_POTENTIAL_DF

,Name,Filename,Model,Species,Config
0,MACE_CuZr,[CuZr_mace_debug.model-lammps.pt],Custom,"[Cu, Zr]","[pair_style mace\n, pair_coeff * * CuZr_mace_d..."


# List pyiron database potentials

In [5]:
# Create a tiny dummy structure (single fcc Cu cell is enough)
dummy_structure = pr.create.structure.bulk("Cu", cubic=True)

tmp = pr.create.job.Lammps("tmp_list_potentials", delete_existing_job=True)
tmp.structure = dummy_structure

pots = tmp.list_potentials()
print("Number of potentials:", len(pots))

# Filter likely Cu–Zr / EAM entries
print("\nCandidates for Cu–Zr EAM:\n")
for p in pots:
    s = p.lower()
    if ("cuzr" in s) or ("cu-zr" in s) or ("mende" in s) or ("eam" in s and ("cu" in s and "zr" in s)):
        print(p)

Number of potentials: 132

Candidates for Cu–Zr EAM:

2007--Mendelev-M-I--Cu-Zr--LAMMPS--ipr1
2008--Kim-Y-M--Cu-Zr--LAMMPS--ipr1
2008--Mendelev-M-I--Cu--LAMMPS--ipr1
2009--Kang-K-H--Cu-Zr-Ag--LAMMPS--ipr1
2009--Mendelev-M-I--Cu-Zr--LAMMPS--ipr1
2012--Mendelev-M-I--Cu--LAMMPS--ipr1
2016--Borovikov-V--Cu-Zr--LAMMPS--ipr1
2019--Mendelev-M-I--Cu-Zr--LAMMPS--ipr1
EAM_Dynamo_BorovikovMendelevKing_2016_CuZr__MO_097471813275_000
EAM_Dynamo_MendelevKing_2013_Cu__MO_748636486270_005
EAM_Dynamo_MendelevKramerBecker_2008_Cu__MO_945691923444_005
EAM_Dynamo_MendelevKramerOtt_2009_CuZr__MO_600021860456_005
EAM_Dynamo_MendelevSordeletKramer_2007_CuZr__MO_120596890176_005
EAM_Dynamo_MendelevSunZhang_2019_CuZr__MO_609260676108_000
EAM_Mendelev_2019_CuZr__MO_945018740343_000
EMT_Asap_MetalGlass_PaduraruKenoufiBailey_2007_CuZr__MO_987541074959_001
MEAM_LAMMPS_KangSaLee_2009_ZrAgCu__MO_813575892799_000


# Defined potentials

In [7]:
EAM_NAMES = [
    "2007--Mendelev-M-I--Cu-Zr--LAMMPS--ipr1",
    "EAM_Mendelev_2019_CuZr__MO_945018740343_000",
    # Optional extra baselines:
    # "2016--Borovikov-V--Cu-Zr--LAMMPS--ipr1",
    # "2008--Kim-Y-M--Cu-Zr--LAMMPS--ipr1",
]

POTENTIALS = [
    {"id": "MACE", "mode": "custom", "df": MACE_POTENTIAL_DF},
]

for name in EAM_NAMES:
    # Make a compact id for filenames
    short_id = name.split("__MO_")[0].replace("--", "_").replace(" ", "_")
    POTENTIALS.append({"id": short_id, "mode": "pyiron", "name": name})

POTENTIALS

[{'id': 'MACE',
  'mode': 'custom',
  'df':         Name                           Filename   Model   Species  \
  0  MACE_CuZr  [CuZr_mace_debug.model-lammps.pt]  Custom  [Cu, Zr]   
  
                                                Config  
  0  [pair_style mace\n, pair_coeff * * CuZr_mace_d...  },
 {'id': '2007_Mendelev-M-I_Cu-Zr_LAMMPS_ipr1',
  'mode': 'pyiron',
  'name': '2007--Mendelev-M-I--Cu-Zr--LAMMPS--ipr1'},
 {'id': 'EAM_Mendelev_2019_CuZr',
  'mode': 'pyiron',
  'name': 'EAM_Mendelev_2019_CuZr__MO_945018740343_000'}]

# One function to assign a potential to a job

In [8]:
def assign_potential(job, pot_spec: dict):
    if pot_spec["mode"] == "custom":
        job.potential = pot_spec["df"]
    elif pot_spec["mode"] == "pyiron":
        job.potential = pot_spec["name"]
    else:
        raise ValueError(f"Unknown pot_spec mode: {pot_spec['mode']}")

# Standard job runners (static + MD)

In [9]:
def make_lammps_job(job_name: str, structure, pot_spec: dict, delete_existing=True, cores=DEFAULT_CORES):
    job = pr.create.job.Lammps(job_name, delete_existing_job=delete_existing)
    job.structure = structure
    assign_potential(job, pot_spec)
    job.server.cores = int(cores)
    return job

def run_static(job_name: str, structure, pot_spec: dict, cores=DEFAULT_CORES):
    job = make_lammps_job(job_name, structure, pot_spec, cores=cores)
    job.calc_static()
    job.run()
    return job

def run_md(job_name: str, structure, pot_spec: dict, T=300, steps=1000, timestep_fs=DEFAULT_TIMESTEP_FS, cores=DEFAULT_CORES):
    job = make_lammps_job(job_name, structure, pot_spec, cores=cores)
    job.calc_md(temperature=T, n_ionic_steps=steps, time_step=timestep_fs)
    job.run()
    return job

# Sanity test 1: static energy for fcc Cu

In [15]:
structure_cu = pr.create.structure.bulk("Cu", cubic=True).repeat([4,4,4])

for pot in POTENTIALS:
    j = run_static(f"sanity_static_Cu_{pot['id']}", structure_cu, pot, cores=1)
    print(pot["id"], "E_pot (last) =", float(j.output.energy_pot[-1]))

The job sanity_static_Cu_MACE was saved and received the ID: 20
MACE E_pot (last) = -899.959247589119
The job sanity_static_Cu_2007_MendelevmMmI_CumZr_LAMMPS_ipr1 was saved and received the ID: 21
2007_Mendelev-M-I_Cu-Zr_LAMMPS_ipr1 E_pot (last) = -839.706694821201
The job sanity_static_Cu_EAM_Mendelev_2019_CuZr was saved and received the ID: 22
EAM_Mendelev_2019_CuZr E_pot (last) = -837.904721690331
